In [1]:
import os
import pandas as pd
import subprocess
import streamlit as st
import re
import boto3
import anthropic
import json
import dspy
import helper as hp
from botocore.exceptions import ClientError
from openai import OpenAI
from mac_vendor_lookup import MacLookup
from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel

load_dotenv()

/home/ash/miniconda3/envs/nanites/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
path_to_pcaps = ["/home/ash/github/packet_analysis/pcap_store/wifi_customer/RoamingIQRadiusfiltered.pcapng"]

session_state = {}
session_state["models"]=("GPT-4o", "Llama-3.3-70b")

#### PCAP to CSV

In [3]:
# Function to convert .pcap to CSV using a subset of fields
def pcap_to_df(pcap_path):
    fields = [
        "frame.number",
        "frame.time",
        "frame.len",
        "frame.ignored",
        "frame.protocols",
        "ip.version",
        "ip.len",
        "ip.src",
        "ip.dst",
        "ip.proto",
        "tcp.srcport",
        "tcp.dstport",
        "tcp.time_relative",
        "tcp.time_delta",
        "tcp.analysis.acks_frame",
        "tcp.analysis.ack_rtt",
        "udp.srcport",
        "udp.dstport",
        "eth.src",
        "eth.src.oui",
        "eth.addr",
        "eth.addr.oui",
        "eth.src.lg",
        "eth.lg",
        "eth.src.ig",
        "eth.ig",
        "eth.dst_resolved",
        "eth.src_resolved",
        "eth.src.oui_resolved",
        "eth.dst.oui",
        "eth.addr",
        "eth.addr.oui",
        "eth.dst.lg",
        "eth.dst.ig",
        "dns.qry.name",
        "dns.a",
        "_ws.expert.message",
    ]
    to_df = hp.PcapToDf(pcap_path)
    curated_df = to_df.create_df()
    #curated_df = df[[col for col in df.columns if col in fields]]
    return curated_df


In [4]:
pcap_df = {}

for pcap in path_to_pcaps:
    file_name = os.path.basename(pcap).split(".")[0]
    print(f"Processing {file_name}")
    df = pcap_to_df(pcap)
    pcap_df[file_name] = df

print("Number of pcaps processed: ", len(pcap_df))

Processing RoamingIQRadiusfiltered
Number of pcaps processed:  1


### print to CSV

In [5]:
csv_store = "/home/ash/github/packet_analysis/csv/"
for key, value in pcap_df.items():
    if not os.path.exists(csv_store):
        os.makedirs(csv_store)
    if f"{key}.csv" not in os.listdir(csv_store):
        value.to_csv(f"{csv_store}{key}.csv", index=False)

In [12]:
def reasoning_logic(lm, context, user_query):
    dspy.configure(lm=lm)
    respond = dspy.ChainOfThought('context, question -> answer')
    result = respond(context=context, question=user_query)
    # Print the history of prompts
    #dspy.inspect_history(n=5)
    return result

def query_interface(user_query, llm):
    """
    Provide an interface to query the processed PCAP table using OpenAI LLM and generate conversational responses.
    """
    dataframe_list = list(pcap_df.values())

    if len(dataframe_list) == 1:
        df_full = dataframe_list[0]
        context = df_full.to_markdown(index=False)
    else:
        context = ""
        for key, value in pcap_df.items():
            df_in_markdown = value.to_markdown(index=False)
            context += f"{key} : {df_in_markdown}\n\n"

    if not user_query.strip():
        st.warning("Please enter a question.")
        return
    try:
        print(f"Generating conversational response with {llm}...")
        if llm==session_state["models"][1]:
            lm = dspy.LM('openai/llama-3.3-70b-versatile', api_key=os.getenv("GROQ_API_KEY"), api_base='https://api.groq.com/openai/v1')
        elif llm==session_state["models"][0]:
            lm = dspy.LM('openai/gpt-4o', api_key=os.getenv("OPENAI_API_KEY"))
        result =reasoning_logic(lm=lm, context=context, user_query=user_query)
        return result.reasoning, result.answer
    except Exception as e:
        st.error(f"Error: {e}")

In [13]:

llm = session_state["models"][0]

In [20]:
import markdown
question = "Draw an ASCII of the pcap file, express in markdown"
result = query_interface(question, llm)

print(result)

Generating conversational response with GPT-4o...
('To create an ASCII representation of the pcap file, we need to consider the sequence of packets and their attributes. The context provides details about each packet, such as source and destination MAC addresses, IP addresses, UDP ports, and timestamps. We can use these details to illustrate the flow of packets between the source and destination addresses. The ASCII diagram will represent each packet as a line connecting the source and destination, with annotations for the packet number, source and destination IPs, and UDP ports.', '```plaintext\nPacket 1: 16:52:6a:41:ce:b3 (73.233.222.192:34032) --> 16:dd:3b:97:52:91 (172.31.65.199:3833)\nPacket 2: 16:dd:3b:97:52:91 (172.31.65.199:3833) --> 16:52:6a:41:ce:b3 (73.233.222.192:34032)\nPacket 3: 16:52:6a:41:ce:b3 (73.233.222.192:39885) --> 16:dd:3b:97:52:91 (172.31.65.199:3834)\nPacket 4: 16:dd:3b:97:52:91 (172.31.65.199:3834) --> 16:52:6a:41:ce:b3 (73.233.222.192:39885)\n```\n\nThis ASCI